# 3D-SynTree V2 — Colab T4 runtime initialization (OPS-01, issue #36)

This notebook initializes the **authorized Google Colab Tesla T4 runtime**
for 3D-SynTree V2 Stage-1 training (branch `3d-synV2`).

**Supported hardware (constitution #45 §5): Google Colab / Kaggle Tesla T4
only. Pure FP32 everywhere — no FP16/BF16/AMP/GradScaler exists in V2.**

Workflow (issue #45 §28–§29):

1. Runtime → Change runtime type → **T4 GPU**, then run the cells in order.
2. Cell 1 verifies the GPU is actually a Tesla T4 (fail-closed otherwise —
   do not train on any other device, #45 §29.2).
3. Cells 2–3 install the system packages and start the SSH daemon for the
   operator's `colab ssh` CLI bridge (the connection commands run in the
   **operator's local terminal**, not here — see `docs/ops/colab_ssh_workflow.md`).
4. Cells 4–6 check out the exact branch commit, install the environment,
   and record the mandatory hardware preflight (GPU VRAM and host RAM
   measured separately, #45 §28.4).
5. Cell 7+ run the canonical Stage-1 training CLI (`scripts/train_stage1.py`)
   — never an invented command (#45 §28.3).

In [ ]:
# Cell 1 — GPU preflight: verify the allocated device is a Tesla T4 (#45 §29.2)
# Fail-closed: any other GPU (or no GPU) must NOT be trained on.
import subprocess, sys

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("torch CUDA:", torch.version.cuda)
assert torch.cuda.is_available(), "NO GPU ALLOCATED — Runtime > Change runtime type > T4 GPU"
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}")
    print(f"total VRAM: {p.total_memory / 1024**3:.2f} GiB")
    print(f"compute capability: {p.major}.{p.minor}")
    is_t4 = p.name == "Tesla T4"
    print(f"is Tesla T4: {is_t4}")

names = [torch.cuda.get_device_properties(i).name for i in range(torch.cuda.device_count())]
assert all(n == "Tesla T4" for n in names), (
    f"NON-T4 GPU ALLOCATED: {names} — the supported production scope is "
    "Tesla T4 only (#45 s5). Report the device and do NOT train on it."
)
free, total = torch.cuda.mem_get_info()
print(f"free VRAM: {free / 1024**3:.2f} GiB of {total / 1024**3:.2f} GiB")
print("PREFLIGHT OK: Tesla T4 confirmed")

In [ ]:
# Cell 2 — system packages (issue #36: aria2, libxrender1, OpenSSH server)
!apt-get update -qq
!apt-get install -y -qq aria2 libxrender1 openssh-server

In [ ]:
# Cell 3 — SSH daemon + public tunnel endpoint (OPS-01, issue #36)
#
# The RUNTIME side of the agent SSH bridge:
#   1. sshd — root login, PUBLIC-KEY ONLY (password auth disabled)
#   2. install the AGENT SANDBOX public key (public identity, not a secret)
#   3. open a public TCP tunnel (bore.pub — no account required) so the
#      agent sandbox can reach this runtime's port 22
#
# Authorization boundary (#45 §29.1): YOU start this runtime + tunnel with
# YOUR account. The agent never authenticates on your behalf. When this
# cell prints ENDPOINT_FOR_AGENT, copy that single line back to the agent;
# it then connects with its private key, which never leaves its sandbox.
# The operator `colab ssh` CLI path (docs/ops/colab_ssh_workflow.md §3)
# remains available as an alternative.
import os
import subprocess
import time

AGENT_PUBKEY = (
    "ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIKnSUjNgNKTSl5ApPZQGrO7UXHN/FyLRCZ3kduwVlHmM"
)  # agent sandbox identity, public half — safe to publish

# --- 1) sshd: public-key-only root login -----------------------------------
os.makedirs("/var/run/sshd", exist_ok=True)
os.makedirs("/root/.ssh", exist_ok=True)
ak = "/root/.ssh/authorized_keys"
existing = open(ak).read() if os.path.exists(ak) else ""
if AGENT_PUBKEY not in existing:
    with open(ak, "a") as handle:
        handle.write(AGENT_PUBKEY + "\n")
os.chmod("/root/.ssh", 0o700)
os.chmod(ak, 0o600)
# first-obtained value wins in sshd: drop-in dir is included FIRST on
# modern Ubuntu; sed the main config too so the policy holds either way
os.makedirs("/etc/ssh/sshd_config.d", exist_ok=True)
with open("/etc/ssh/sshd_config.d/00-syntree-bridge.conf", "w") as handle:
    handle.write(
        "PermitRootLogin prohibit-password\n"
        "PubkeyAuthentication yes\n"
        "PasswordAuthentication no\n"
        "KbdInteractiveAuthentication no\n"
    )
subprocess.run(
    "sed -i "
    r"'s/^#\?PermitRootLogin.*/PermitRootLogin prohibit-password/; "
    r"s/^#\?PasswordAuthentication.*/PasswordAuthentication no/; "
    r"s/^#\?KbdInteractiveAuthentication.*/KbdInteractiveAuthentication no/' "
    "/etc/ssh/sshd_config",
    shell=True, check=False,
)
subprocess.run(["pkill", "-x", "sshd"], check=False)  # restart cleanly
subprocess.run(["/usr/sbin/sshd"], check=False)
print("sshd: started (root, public-key only; agent key installed)")

# --- 2) public TCP tunnel: bore.pub (no account) -----------------------------
os.makedirs("/content/logs", exist_ok=True)
BORE_URL = (
    "https://github.com/ekzhang/bore/releases/download/v0.6.0/"
    "bore-v0.6.0-x86_64-unknown-linux-musl.tar.gz"
)
if not os.path.exists("/usr/local/bin/bore"):
    subprocess.run(["wget", "-q", BORE_URL, "-O", "/tmp/bore.tgz"], check=True)
    subprocess.run(
        ["tar", "-xzf", "/tmp/bore.tgz", "-C", "/usr/local/bin", "bore"],
        check=True,
    )
subprocess.run(["pkill", "-f", "bore local 22"], check=False)  # stale tunnels
with open("/content/logs/bore.log", "w") as logf:
    subprocess.Popen(
        ["/usr/local/bin/bore", "local", "22", "--to", "bore.pub"],
        stdout=logf, stderr=subprocess.STDOUT,
    )
time.sleep(5)  # allow the control handshake
log = open("/content/logs/bore.log").read()
print("--- bore.log ---")
print(log)
endpoints = [ln.split()[-1] for ln in log.splitlines() if "bore.pub:" in ln]
if endpoints:
    print("ENDPOINT_FOR_AGENT =", endpoints[-1])
    print("^^ copy this line to the agent (host:port)")
else:
    print("bore tunnel did not come up — run Cell 3b (ngrok fallback)")


In [ ]:
# Cell 3b — tunnel fallback / status / reconnect (OPS-01, #36)
#
# Use this cell if:
#   - Cell 3's bore tunnel failed, or
#   - the endpoint went stale and you need to re-expose port 22, or
#   - you prefer ngrok (requires YOUR free ngrok authtoken:
#     https://dashboard.ngrok.com/get-started/your-authtoken)
#
# Re-running this cell re-checks/re-establishes the tunnel and prints the
# current ENDPOINT_FOR_AGENT line. The runtime + training keep running
# regardless: the tunnel only carries the agent's SSH session.
import os
import subprocess
import time

NGROK_AUTHTOKEN = ""  # paste YOUR ngrok authtoken to use the ngrok path


def bore_status():
    try:
        log = open("/content/logs/bore.log").read()
    except FileNotFoundError:
        return None
    eps = [ln.split()[-1] for ln in log.splitlines() if "bore.pub:" in ln]
    alive = subprocess.run(
        ["pgrep", "-f", "bore local 22"], capture_output=True
    ).returncode == 0
    return ("alive" if alive else "DEAD", eps[-1] if eps else None)


if NGROK_AUTHTOKEN:
    subprocess.run(["pip", "install", "-q", "pyngrok"], check=True)
    from pyngrok import ngrok  # noqa: E402
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    for t in list(ngrok.get_tunnels()):
        ngrok.disconnect(t.public_url)
    tunnel = ngrok.connect(22, "tcp")
    print("ENDPOINT_FOR_AGENT =", tunnel.public_url.replace("tcp://", ""))
    print("^^ copy this line to the agent (host:port)")
else:
    st = bore_status()
    if st is None:
        print("no bore log — run Cell 3 first, or set NGROK_AUTHTOKEN")
    elif st[0] == "alive":
        print("bore tunnel alive; ENDPOINT_FOR_AGENT =", st[1])
    else:
        print("bore tunnel DEAD — restarting it")
        subprocess.run(["pkill", "-f", "bore local 22"], check=False)
        with open("/content/logs/bore.log", "w") as logf:
            subprocess.Popen(
                ["/usr/local/bin/bore", "local", "22", "--to", "bore.pub"],
                stdout=logf, stderr=subprocess.STDOUT,
            )
        time.sleep(5)
        log = open("/content/logs/bore.log").read()
        eps = [ln.split()[-1] for ln in log.splitlines() if "bore.pub:" in ln]
        print("ENDPOINT_FOR_AGENT =", eps[-1] if eps else "STILL DOWN — use ngrok")


In [ ]:
# Cell 4 — repository checkout: exact branch + commit (#45 §28.3 / §29.3)
%cd /content
![ -d 3d-syntree ] || git clone -b 3d-synV2 https://github.com/Vtheonly/3d-syntree.git
%cd /content/3d-syntree
!git fetch origin 3d-synV2
!git status --short
!git rev-parse HEAD
!git log --oneline -3

In [ ]:
# Cell 5 — Python environment from the repository's declared spec (#45 §29.3)
# Colab ships torch + CUDA preinstalled: verify, never reinstall blindly.
import torch
print("preinstalled torch:", torch.__version__, "| CUDA:", torch.version.cuda)
%pip install -q -e ".[test]"
%pip install -q torch_geometric h5py pandas pyarrow psutil
import rdkit, torch_geometric
print("rdkit:", rdkit.__version__, "| pyg:", torch_geometric.__version__)

In [ ]:
# Cell 6 — mandatory hardware preflight: host RAM and GPU VRAM measured
# SEPARATELY, plus disk (issue #45 §28.4 / §29.2; recorded verbatim).
import json, subprocess

for cmd in (["free", "-h"], ["df", "-h", "/content"]):
    print("$", " ".join(cmd))
    print(subprocess.run(cmd, capture_output=True, text=True).stdout)

import psutil, torch
vm = psutil.virtual_memory()
preflight = {
    "gpu": {
        "name": torch.cuda.get_device_name(0),
        "total_vram_mib": round(torch.cuda.get_device_properties(0).total_memory / 1024**2, 1),
        "free_vram_mib": round(torch.cuda.mem_get_info()[0] / 1024**2, 1),
        "compute_capability": f"{torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}",
        "torch_allocated_mib": round(torch.cuda.memory_allocated() / 1024**2, 1),
        "torch_reserved_mib": round(torch.cuda.memory_reserved() / 1024**2, 1),
    },
    "host": {
        "total_ram_mib": round(vm.total / 1024**2, 1),
        "available_ram_mib": round(vm.available / 1024**2, 1),
        "used_ram_mib": round(vm.used / 1024**2, 1),
        "process_rss_mib": round(psutil.Process().memory_info().rss / 1024**2, 1),
        "cpu_count": psutil.cpu_count(),
    },
}
print(json.dumps(preflight, indent=2))
with open("/content/hardware_preflight.json", "w") as handle:
    json.dump(preflight, handle, indent=2)

In [ ]:
# Cell 7 — artifacts + GPU forward/backward smoke test (#45 §28.5 steps 3-4)
# The canonical battery is Suite-B 4 (tests/suite_b/test_t4_memory_boundary.py):
# the B=12 x cap-700 full-policy Profile-A forward/backward with peak reserved
# VRAM asserted against the 9.5 GiB budget (#26/#33). It exercises the REAL
# SynTreePolicy end-to-end — a hand-rolled partial forward proves less.
import os, subprocess, sys

CATALOG = os.environ.get("SYNTREE_CATALOG_H5", "/content/work/catalog/enamine_real_50k_pharm3d_d256.h5")
print("catalog:", CATALOG, "| exists:", os.path.exists(CATALOG))

env = {**os.environ, "SYNTREE_CATALOG_H5": CATALOG}
r = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/suite_b/test_t4_memory_boundary.py",
     "-v", "--no-header"],
    cwd="/content/3d-syntree", capture_output=True, text=True, env=env,
)
print(r.stdout[-4000:])
assert r.returncode == 0, "Suite-B T4 memory-boundary battery FAILED"
print("SMOKE OK: full-profile forward/backward within the 9.5 GiB budget")


In [ ]:
# Cell 8 — the canonical Stage-1 training launch (#26; #45 §28.5 step 6)
# The exact repository CLI — never an invented command. Adjust the paths
# to the verified artifact locations; use --time-budget-seconds for a
# controlled graceful stop (session chaining, #43).
!python scripts/train_stage1.py \
    --config configs/train_colab_t4.json \
    --trajectories /content/work/data/trajectories \
    --processed /content/work/data/processed \
    --catalog "$SYNTREE_CATALOG_H5" \
    --run-dir /content/runs/stage1_t4 \
    --device cuda \
    --time-budget-seconds 8400

In [ ]:
# Cell 9 — training telemetry tail (host RAM and GPU VRAM separately; #45 §28.6)
!tail -n 3 /content/runs/stage1_t4/telemetry.jsonl 2>/dev/null || echo "no telemetry yet"
!ls -la /content/runs/stage1_t4/ 2>/dev/null || true

## Operator notes (the authorization boundary, #45 §29.1)

Two equivalent bridges exist; either is sufficient:

**Path A — agent-sandbox bridge (default).** Run Cells 1–3 in order.
Cell 3 prints `ENDPOINT_FOR_AGENT = host:port` (a bore.pub TCP tunnel
opened with your Colab session — no extra account needed). Copy that one
line back to the agent and confirm the runtime is authorized. The agent
then connects with its sandbox keypair (private key never leaves the
sandbox; only the public half is embedded here). If bore is unreachable,
run Cell 3b with your ngrok authtoken instead. If the tunnel drops later,
re-run Cell 3b to print a fresh endpoint — the runtime and any training
process keep running.

**Path B — operator-local `colab` CLI.** In your local terminal run
`colab ssh` (or `ssh -o ProxyCommand="colab ssh --proxy-mode -s %h" SESSION`),
open the link Colab produces, authenticate with your own Google account,
and confirm authorization back to the agent.

The agent never fabricates links, never authenticates as you, and never
asks for passwords or private keys in chat (#45 §29.1).

After the bridge is up, continue with Cells 4–8 (checkout → env →
preflight → smoke → training). The agent drives its own verification
over the bridge via `scripts/colab_ssh_client.py` (§29.2 battery).
